In [ ]:
import os
os.chdir(path_to_wd)

import sys
import pickle
import anndata as ad
import scanpy as sc
import pandas as pd
from pandas.api.types import CategoricalDtype
import numpy as np
import scipy
import scipy.sparse as sp
from scipy import sparse
import tacco as tc
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import cm, colors
import pycats
from pandas.api.types import CategoricalDtype

plt.rcParams['font.family']= 'Dejavu serif'
plt.rcParams['pdf.fonttype'] = 'truetype'
matplotlib.rcParams['figure.dpi'] = 100

In [ ]:
## Plotting options
highres = True
default_dpi = 200.0
if highres:
    matplotlib.rcParams['figure.dpi'] = 648.0
    hr_ext = '_hd'
else:
    matplotlib.rcParams['figure.dpi'] = default_dpi
    hr_ext = ''

axsize = np.array([4,3])*0.5

region_colors = tc.pl.get_default_colors([f'region_{i}' for i in range(4)], offset=17)
split_names = np.array([f'sub_{i}' for i in range(4)])
split_colors = tc.pl.get_default_colors(split_names, offset=12)

vega_20 = list(map(colors.to_hex, cm.tab20.colors))

In [ ]:
palette_clone = ['#9fc3e7', '#f9d6a5', '#91cb98', '#f19b98', '#c5b7d9', '#dab89a', '#e6aece', '#b0dac7', 
                 '#d2bf1b', "#BFA4D6", '#f4b1ab', "#94b0cb", '#c9e3c1', '#dbc9e1', '#f5b081', "#acdad2",
                 '#b61d23', '#ec671a', '#7c3c70', '#11793b', '#005084'] 

In [ ]:
# color map
from matplotlib.colors import ListedColormap, LinearSegmentedColormap

blue_yellow_colors = ["#352A86","#343DAE","#0262E0","#1389D2","#2DB7A3","#A5BE6A","#F8BA43","#F6DA23","#F8FA0D"]
horizon_colors = ["#000033","#000075","#0000B6","#0000F8","#2E00FF","#6100FF","#9408F7","#C729D6","#FA4AB5",
                 "#FF6A95","#FF8B74","#FFAC53","#FFCD32","#FFEE11","#FFFF60"]
solar_extra_colors = ["#3361A5","#248AF3","#14B3FF","#88CEEF","#C1D5DC","#EAD397","#FDB31A","#E42A2A","#A31D1D"]

blue_yellow = LinearSegmentedColormap.from_list('custom',blue_yellow_colors)
horizon = LinearSegmentedColormap.from_list('custom',horizon_colors)
solar_extra = LinearSegmentedColormap.from_list('custom',solar_extra_colors)

## Data Loading

In [ ]:
adata_sp = sc.read("Reproducibility/Data/TREKKER/UC_TREKKER_RNA_ATAC_spatial.h5ad")

In [ ]:
for sample, df in adata_sp.obs.groupby('sample'):
    x_min = df['x'].min()
    y_min = df['y'].min()
    adata_sp.obs.loc[df.index, 'x'] = df['x'] - x_min
    adata_sp.obs.loc[df.index, 'y'] = df['y'] - y_min

In [ ]:
lineage_colors = {'TNK': '#C10020', 'B': '#FF6800', 'Endothelial': '#00538A', 
                  'Myeloid': '#803E75', 'CAF':'#007D34', 'Epithelial': '#E6BD8A'}

# utility function for plotting
def state_plot_grid_S(padding=None):
    # fixed mapping of sample to axis to sort by state
    fig,axs = tc.pl.subplots(3,3,x_padding=padding,y_padding=padding)
    state_axes = np.empty_like(axs, shape=(1,9))
    state_axes[0,0] = axs[0,0]
    state_axes[0,1] = axs[0,1]
    state_axes[0,2] = axs[0,2]
    state_axes[0,3] = axs[1,0]
    state_axes[0,4] = axs[1,1]
    state_axes[0,5] = axs[1,2]
    state_axes[0,6] = axs[2,0]
    state_axes[0,7] = axs[2,1]
    state_axes[0,8] = axs[2,2]
    return state_axes[:, :9]

# utility function for plotting
def state_plot_grid_L(padding=None):
    # fixed mapping of sample to axis to sort by state
    fig,axs = tc.pl.subplots(2,1,x_padding=padding,y_padding=padding)
    state_axes = np.empty_like(axs, shape=(1,2))
    state_axes[0,0] = axs[0,0]
    state_axes[0,1] = axs[0,1]
    return state_axes[:, :2] 


In [ ]:
cat_type = CategoricalDtype(categories=['P01_clone_1', 
            'P02_clone_1', 'P02_clone_2', 
            'P03_clone_1', 
            'P04_clone_1', 'P04_clone_2', 
            'P05_clone_1', 'P05_clone_2', 
            'P06_clone_1', 'P06_clone_2', 'P06_clone_3', 'P06_clone_4', 
            'P07_clone_1', 'P07_clone_2', 
            'P09_clone_1', 'P09_clone_2', 
            'TNK','B','Myeloid','CAF','Endothelial'], ordered=True)
adata_sp.obs['lineage_w_clone'] = adata_sp.obs['lineage_w_clone'].astype(cat_type)

In [ ]:
cat_type = CategoricalDtype(categories=['P01_clone_1', 
            'P02_clone_1_hypoxic', 'P02_clone_1','P02_clone_2', 
            'P03_clone_1', 
            'P04_clone_1', 'P04_clone_2', 
            'P05_clone_1', 'P05_clone_2', 
            'P06_clone_1_stress', 'P06_clone_1', 'P06_clone_2_stress', 'P06_clone_2', 
            'P06_clone_3', 'P06_clone_4', 
            'P07_clone_1', 'P07_clone_2', 
            'P09_clone_1', 'P09_clone_2', 
            'TNK','B','Myeloid','CAF','Endothelial'], ordered=True)
adata_sp.obs['lineage_w_clone2'] = adata_sp.obs['lineage_w_clone2'].astype(cat_type)

In [ ]:
adata_sp_L = adata_sp[adata_sp.obs['sample'].isin(['P02','P06'])==True].copy()
adata_sp_S = adata_sp[adata_sp.obs['sample'].isin(['P02','P06'])==False].copy()

## Spatial plot

- lineage w clone

In [ ]:
labels = adata_sp.obs['lineage_w_clone'].cat.categories.tolist()
color_dict = dict(zip(labels, palette_clone))

pucks = adata_sp_L.copy()
tdatas = {sample: pucks[df.index] for sample, df in pucks.obs.groupby('specimen') }
axs = state_plot_grid_L(padding=1.5)
fig = tc.pl.scatter(tdatas,
                    keys="lineage_w_clone",
                    colors=color_dict,
                    position_key=['x','y'],
                    joint=True, point_size=4, 
                    ax=axs, 
                    noticks=False,
                    legend=True,
                    rasterized=True
                    )
plt.savefig(f"Reproducibility/Results/Plots/Slide-tags/FigureS12F_L.pdf", bbox_inches='tight')
plt.close()

In [ ]:
labels = adata_sp.obs['lineage_w_clone'].cat.categories.tolist()
color_dict = dict(zip(labels, palette_clone))

pucks = adata_sp_S.copy()
tdatas = {sample: pucks[df.index] for sample, df in pucks.obs.groupby('specimen') }
axs = state_plot_grid_S(padding=1.5)
fig = tc.pl.scatter(tdatas,
                    keys="lineage_w_clone",
                    colors=color_dict,
                    position_key=['x','y'],
                    joint=True, point_size=20, 
                    ax=axs, 
                    noticks=False,
                    legend=True,
                    rasterized=True
                    )
plt.savefig(f"Reproducibility/Results/Plots/Slide-tags/FigureS12F_S.pdf", bbox_inches='tight')
plt.close()

- P02

In [ ]:
palette_clone3 = ['#d3d3d3', '#c64f1b', '#f9d6a5', '#91cb98', '#d3d3d3', '#d3d3d3', '#d3d3d3', '#d3d3d3',
                  '#d3d3d3', '#d3d3d3', '#d3d3d3', "#d3d3d3", '#d3d3d3', '#d3d3d3', '#d3d3d3', '#d3d3d3', 
                  '#d3d3d3', '#d3d3d3', '#d3d3d3', '#d3d3d3', '#d3d3d3', '#d3d3d3', '#d3d3d3', '#005084']

labels = adata_sp.obs['lineage_w_clone2'].cat.categories.tolist()
color_dict = dict(zip(labels, palette_clone3))

In [ ]:
pucks = adata_sp_L.copy()

# original tdatas & axes
tdatas = {sample: pucks[df.index] for sample, df in pucks.obs.groupby('specimen')}
samples = list(tdatas.keys())

axs = state_plot_grid_L(padding=1.5)

# 1) background: all cells
fig = tc.pl.scatter(
    tdatas,
    keys="lineage_w_clone2",
    colors=color_dict,
    position_key=['x', 'y'],
    joint=True,
    point_size=4,
    ax=axs,
    noticks=False,
    legend=True,
    rasterized=False,
    render=False
)

highlight_cats = ['P02_clone_1_hypoxic']
axs_flat = np.array(axs).ravel()

for i, sample in enumerate(samples):
    ad_s = tdatas[sample]
    obs = ad_s.obs
    mask = obs['lineage_w_clone2'].isin(highlight_cats)
    if not mask.any():
        continue  # no highlight cells in this specimen
    ax_s = axs_flat[i]
    # draw each highlighted category with its own color, on top
    for cat in obs.loc[mask, 'lineage_w_clone2'].unique():
        sub = obs[mask & (obs['lineage_w_clone2'] == cat)]
        ax_s.scatter(
            sub['x'],
            sub['y'],
            s=4,                             # a bit larger
            c=color_dict[cat],               # same color as in color_dict
            linewidths=0,
            zorder=10,                       # ensure on top
        )

plt.savefig("Reproducibility/Results/Plots/Slide-tags/Figure7F.pdf",
            bbox_inches="tight")
plt.close()

- P06

In [ ]:
palette_clone4 = ['#d3d3d3', '#d3d3d3', "#d3d3d3", '#d3d3d3', '#d3d3d3', '#d3d3d3', '#d3d3d3', '#d3d3d3',
                  '#d3d3d3', "#f71804", '#d2bf1b', "#b12e23", "#BFA4D6", '#f4b1ab', "#94b0cb", '#d3d3d3', 
                  '#d3d3d3', "#d3d3d3", '#d3d3d3', "#d3d3d3", '#d3d3d3', '#d3d3d3', '#d3d3d3', '#00538A']

labels = adata_sp.obs['lineage_w_clone2'].cat.categories.tolist()
color_dict = dict(zip(labels, palette_clone4))

In [ ]:
pucks = adata_sp_L.copy()

# original tdatas & axes
tdatas = {sample: pucks[df.index] for sample, df in pucks.obs.groupby('specimen')}
samples = list(tdatas.keys())

axs = state_plot_grid_L(padding=1.5)

# 1) background: all cells
fig = tc.pl.scatter(
    tdatas,
    keys="lineage_w_clone2",
    colors=color_dict,
    position_key=['x', 'y'],
    joint=True,
    point_size=4,
    ax=axs,
    noticks=False,
    legend=True,
    rasterized=False,
    render=False
)

highlight_cats = ['P06_clone_1_stress', 'P06_clone_2_stress']
axs_flat = np.array(axs).ravel()

for i, sample in enumerate(samples):
    ad_s = tdatas[sample]
    obs = ad_s.obs
    mask = obs['lineage_w_clone2'].isin(highlight_cats)
    if not mask.any():
        continue  # no highlight cells in this specimen
    ax_s = axs_flat[i]
    # draw each highlighted category with its own color, on top
    for cat in obs.loc[mask, 'lineage_w_clone2'].unique():
        sub = obs[mask & (obs['lineage_w_clone2'] == cat)]
        ax_s.scatter(
            sub['x'],
            sub['y'],
            s=4,                             # a bit larger
            c=color_dict[cat],               # same color as in color_dict
            linewidths=0,
            zorder=10,                       # ensure on top
        )

plt.savefig("Reproducibility/Results/Plots/Slide-tags/FigureS13E_clone.pdf",
            bbox_inches="tight")
plt.close()


## Co-occurence analysis

- P02

In [ ]:
puck = adata_sp[adata_sp.obs['specimen']=='P02'].copy()
puck = puck[puck.obs['lineage'].isin(['LUM','Endothelial'])]

tc.tl.co_occurrence_matrix(puck,
                           annotation_key='lineage_w_clone2',
                           result_key='lineage_w_clone2-lineage_w_clone2',
                           max_distance=50,n_permutation=50)

In [ ]:
# Export to R

z = puck.uns['lineage_w_clone2-lineage_w_clone2']["z"]
z_matrix = z.squeeze()
cats = puck.obs['lineage_w_clone2'].cat.categories.tolist()

# Create a DataFrame
z_df = pd.DataFrame(z_matrix, index=cats, columns=cats)
z_df.to_csv(f'Reproducibility/Results/TREKKER/co_occurrence_matrix_P02_zscore_distance_50.txt', sep='\t')

- P06

In [ ]:
puck = adata_sp[adata_sp.obs['specimen']=='P06'].copy()
puck = puck[puck.obs['lineage'].isin(['B'])==False]

tc.tl.co_occurrence_matrix(puck,
                           annotation_key='lineage_w_clone2',
                           result_key='lineage_w_clone2-lineage_w_clone2',
                           max_distance=50,n_permutation=50)

In [ ]:
# Export to R

z = puck.uns['lineage_w_clone2-lineage_w_clone2']["z"]
z_matrix = z.squeeze()
cats = puck.obs['lineage_w_clone2'].cat.categories.tolist()

# Create a DataFrame
z_df = pd.DataFrame(z_matrix, index=cats, columns=cats)
z_df.to_csv(f'Reproducibility/Results/TREKKER/co_occurrence_matrix_P06_zscore_distance_50.txt', sep='\t')

## Spatial plot (Malignant cells)

In [ ]:
adata_m = adata_sp[adata_sp.obs['lineage'].isin(['LUM','SQM','NRP'])].copy()

In [ ]:
sig_df = pd.read_csv("Reproducibility/Results/VISIONR/UC_TREKKER_Malignant_signature_score_literature.txt", index_col=0, sep = '\t')
sig_df.columns = sig_df.columns.str.replace('/', '_', regex=False)

adata_m.obs = adata_m.obs.merge(sig_df.loc[adata_m.obs_names,:], left_index=True, right_index=True, how='left')

### Signature score

- P02 hypoxia

In [ ]:
puck = adata_m[adata_m.obs['specimen'].isin(['P02'])].copy()
signature = 'Hypoxia'

fig = tc.pl.scatter(puck,
                    keys=[signature],
                    cmap = 'Reds',
                    cmap_vmin_vmax=(0.1, 0.8),
                    position_key=['x', 'y'],
                    joint=True, 
                    point_size=4, 
                    noticks=False,
                    legend=True,
                    rasterized=True
                    )
plt.savefig("Reproducibility/Results/Plots/Slide-tags/Figure7E.pdf",
            bbox_inches="tight")
plt.close()

- P06 stress

In [ ]:
puck = adata_m[adata_m.obs['specimen'].isin(['P06'])].copy()
signature = 'Stress_Barkley'

fig = tc.pl.scatter(puck,
                    keys=[signature],
                    cmap = 'Reds',
                    cmap_vmin_vmax=(0.3, 0.7),
                    position_key=['x', 'y'],
                    joint=True, 
                    point_size=4, 
                    noticks=False,
                    legend=True,
                    rasterized=True
                    )
plt.savefig("Reproducibility/Results/Plots/Slide-tags/FigureS13E_signature.pdf",
            bbox_inches="tight")
plt.close()

### TF activity

- P02 

In [ ]:
tmp_path = "Reproducibility/Results/LINGER/TREKKER/output/cell_population_TF_activity_zscore_P02.txt"
TF_activity = pd.read_csv(tmp_path, index_col=0, sep="\t")

cell_use = np.intersect1d(adata_sp.obs_names, TF_activity.index)
adata_sp_tmp = adata_sp[cell_use,:].copy()

adata_sp_tmp.obs = adata_sp_tmp.obs.merge(TF_activity.loc[adata_sp_tmp.obs_names,:], left_index=True, right_index=True, how='left')

In [ ]:
tc.pl.scatter(adata_sp_tmp,
              keys=['HIF1A','NFE2L2','GATA3','FOXA1','MYCN'],
              cmap=solar_extra,
              cmap_vmin_vmax=(-1, 1),
              position_key=['x', 'y'],
              joint=True, point_size=4, 
              noticks=False,
              legend=True,
              rasterized=True
              )
plt.savefig(f"Reproducibility/Results/Plots/Slide-tags/Figure7K_and_S13D.pdf", bbox_inches='tight', dpi=648)
plt.close()

- P06

In [ ]:
tmp_path = "Reproducibility/Results/LINGER/TREKKER/output/cell_population_TF_activity_zscore_P06.txt"
TF_activity = pd.read_csv(tmp_path, index_col=0, sep="\t")

cell_use = np.intersect1d(adata_sp.obs_names, TF_activity.index)
adata_sp_tmp = adata_sp[cell_use,:].copy()

adata_sp_tmp.obs = adata_sp_tmp.obs.merge(TF_activity.loc[adata_sp_tmp.obs_names,:], left_index=True, right_index=True, how='left')

In [ ]:
tc.pl.scatter(adata_sp_tmp,
              keys=['KLF4','FOS'],
              cmap=solar_extra,
              cmap_vmin_vmax=(-1, 1),
              position_key=['x', 'y'],
              joint=True, point_size=4, 
              noticks=False,
              legend=True,
              rasterized=True
              )
plt.savefig(f"Reproducibility/Results/Plots/Slide-tags/FigureS13H.pdf", bbox_inches='tight')
plt.close()